In [ ]:
import h5py 
import numpy as np
import math 
import cv2
import tqdm 

In [2]:
# h5py_path_libero = "/home/ns1254/LIBERO/libero/datasets/libero_90/KITCHEN_SCENE1_open_the_bottom_drawer_of_the_cabinet_demo.hdf5"
h5py_path_libero = "/home/ns1254/openvla/LIBERO/libero/datasets/libero_90_no_noops/LIVING_ROOM_SCENE6_put_the_red_mug_on_the_plate_demo.hdf5"
h5py_path_franka = "/home/ns1254/data_franka/coffee_pod_sujosh70.hdf5"
# h5py_path_franka = "/home/ns1254/dataset_paper3_old/franka/microwave_expert_rick.hdf5"

In [3]:
file_libero = h5py.File(h5py_path_libero, 'r')
file_franka = h5py.File(h5py_path_franka, 'r')

In [4]:
demo_libero=file_libero['data/demo_0']
demo_franka=file_franka['data/demo_0']

In [5]:
demo_libero.keys()

<KeysViewHDF5 ['actions', 'dones', 'obs', 'rewards', 'robot_states', 'states']>

In [6]:
demo_franka.keys()

<KeysViewHDF5 ['actions', 'obs']>

In [7]:
demo_libero['actions'].shape, demo_franka['actions'].shape

((116, 7), (189, 7))

In [8]:
for key in demo_libero['obs'].keys():
    print(f"key: {key} shape: {demo_libero['obs'][key].shape}")

key: agentview_rgb shape: (116, 256, 256, 3)
key: ee_ori shape: (116, 3)
key: ee_pos shape: (116, 3)
key: ee_states shape: (116, 6)
key: eye_in_hand_rgb shape: (116, 256, 256, 3)
key: gripper_states shape: (116, 2)
key: joint_states shape: (116, 7)


In [9]:
for key in demo_franka['obs'].keys():
    print(f"key: {key} shape: {demo_franka['obs'][key].shape}")

key: agentview_rgb shape: (189, 120, 160, 3)
key: ee_states shape: (189, 16)
key: eye_in_hand_rgb shape: (189, 120, 160, 3)
key: gripper_states shape: (189, 1)
key: joint_states shape: (189, 7)


In [10]:
# https://github.com/ARISE-Initiative/robosuite/blob/eafb81f54ffc104f905ee48a16bb15f059176ad3/robosuite/utils/transform_utils.py

def mat2quat(rmat):
    """
    Converts given rotation matrix to quaternion.

    Args:
        rmat (np.array): 3x3 rotation matrix

    Returns:
        np.array: (x,y,z,w) float quaternion angles
    """
    M = np.asarray(rmat).astype(np.float32)[:3, :3]

    m00 = M[0, 0]
    m01 = M[0, 1]
    m02 = M[0, 2]
    m10 = M[1, 0]
    m11 = M[1, 1]
    m12 = M[1, 2]
    m20 = M[2, 0]
    m21 = M[2, 1]
    m22 = M[2, 2]
    # symmetric matrix K
    K = np.array(
        [
            [m00 - m11 - m22, np.float32(0.0), np.float32(0.0), np.float32(0.0)],
            [m01 + m10, m11 - m00 - m22, np.float32(0.0), np.float32(0.0)],
            [m02 + m20, m12 + m21, m22 - m00 - m11, np.float32(0.0)],
            [m21 - m12, m02 - m20, m10 - m01, m00 + m11 + m22],
        ]
    )
    K /= 3.0
    # quaternion is Eigen vector of K that corresponds to largest eigenvalue
    w, V = np.linalg.eigh(K)
    inds = np.array([3, 0, 1, 2])
    q1 = V[inds, np.argmax(w)]
    if q1[0] < 0.0:
        np.negative(q1, q1)
    inds = np.array([1, 2, 3, 0])
    return q1[inds]
    
def mat2pose(hmat):
    """
    Converts a homogeneous 4x4 matrix into pose.

    Args:
        hmat (np.array): a 4x4 homogeneous matrix

    Returns:
        2-tuple:

            - (np.array) (x,y,z) position array in cartesian coordinates
            - (np.array) (x,y,z,w) orientation array in quaternion form
    """
    pos = hmat[:3, 3]
    orn = mat2quat(hmat[:3, :3])
    return pos, orn
    
def mat4(array):
    """
    Converts an array to 4x4 matrix.

    Args:
        array (n-array): the array in form of vec, list, or tuple

    Returns:
        np.array: a 4x4 numpy matrix
    """
    return np.array(array, dtype=np.float32).reshape((4, 4))


def quat2axisangle(quat):
    """
    Converts quaternion to axis-angle format.
    Returns a unit vector direction scaled by its angle in radians.

    Args:
        quat (np.array): (x,y,z,w) vec4 float angles

    Returns:
        np.array: (ax,ay,az) axis-angle exponential coordinates
    """
    # clip quaternion
    if quat[3] > 1.0:
        quat[3] = 1.0
    elif quat[3] < -1.0:
        quat[3] = -1.0

    den = np.sqrt(1.0 - quat[3] * quat[3])
    if math.isclose(den, 0.0):
        # This is (close to) a zero degree rotation, immediately return
        return np.zeros(3)

    return (quat[:3] * 2.0 * math.acos(quat[3])) / den

In [11]:
def franka_ee_states_to_pos_quat(ee_states):
    ee_states = demo_franka['obs']['ee_states'][:]  # (N, 16)
    T_all = np.array([mat4(ee_states[i]) for i in range(ee_states.shape[0])])  # (N, 4, 4)
    pos_all=[]
    ori_all=[]
    for i in range(T_all.shape[0]):
        pos, orn = mat2pose(T_all[i])
        pos_all.append(pos)
        ori_all.append(orn)
    pos_all = np.array(pos_all)
    quat_all = np.array(ori_all)
    return pos_all, quat_all

In [12]:
def franka_obs_to_libero_like_obs(franka_obs):
    joint_states = franka_obs['joint_states'][:]  # (N, 7)
    gripper_states = franka_obs['gripper_states'][:]  # (N, 1) 
    gripper_states = np.repeat(gripper_states, repeats=2, axis=1)  # (N, 2)

    # ee_states from 16d vector to pos + rpy
    ee_states = franka_obs['ee_states'][:]  # (N, 16)
    pos_all, quat_all = franka_ee_states_to_pos_quat(ee_states) 
    rpy_all = np.array([quat2axisangle(quat_all[i]) for i in range(quat_all.shape[0])]) 
    ee_states = np.hstack([pos_all, rpy_all] )

    # convert images to 256x256
    img = franka_obs['agentview_rgb'][:]  
    img_wrist = franka_obs['eye_in_hand_rgb'][:]
    img = np.array([cv2.resize(img[i], (256,256)) for i in range(img.shape[0])])
    img_wrist = np.array([cv2.resize(img_wrist[i], (256,256)) for i in range(img_wrist.shape[0])])

    libero_like_obs = {
        'ee_states': ee_states,  # (N, 6)
        'ee_pos': ee_states[:, :3],  # (N, 3)
        'ee_ori': ee_states[:, 3:],  # (N, 3)
        'gripper_states': gripper_states,  # (N, 2)
        'joint_states': joint_states,  # (N, 7)
        'agentview_rgb': img,  # (N, 256, 256, 3)
        'eye_in_hand_rgb': img_wrist,  # (N, 256, 256, 3)
    }
    return libero_like_obs
    

In [13]:
libero_like_obs = franka_obs_to_libero_like_obs(demo_franka['obs'])
for key in libero_like_obs.keys():
    print(f"key: {key} shape: {libero_like_obs[key].shape}")

key: ee_states shape: (189, 6)
key: ee_pos shape: (189, 3)
key: ee_ori shape: (189, 3)
key: gripper_states shape: (189, 2)
key: joint_states shape: (189, 7)
key: agentview_rgb shape: (189, 256, 256, 3)
key: eye_in_hand_rgb shape: (189, 256, 256, 3)


In [14]:
for key in demo_libero['obs'].keys():
    print(f"key: {key} shape: {demo_libero['obs'][key].shape}")

key: agentview_rgb shape: (116, 256, 256, 3)
key: ee_ori shape: (116, 3)
key: ee_pos shape: (116, 3)
key: ee_states shape: (116, 6)
key: eye_in_hand_rgb shape: (116, 256, 256, 3)
key: gripper_states shape: (116, 2)
key: joint_states shape: (116, 7)


In [15]:
new_data_path =h5py_path_franka.replace(".hdf5", "")+"_regen_libero_like.hdf5"
new_data_file = h5py.File(new_data_path, "w")
grp = new_data_file.create_group("data")

In [16]:
demo_names = list(file_franka['data'].keys())
for i, demo_name in enumerate(demo_names):

    demo_franka=file_franka[f'data/{demo_name}']

    actions = demo_franka['actions'][:]
    libero_like_obs = franka_obs_to_libero_like_obs(demo_franka['obs'])

    ep_data_grp = grp.create_group(f"demo_{i}")
    ep_data_grp.create_dataset("actions", data=actions)
    obs_grp = ep_data_grp.create_group("obs") 
    for key in libero_like_obs.keys():
        obs_grp.create_dataset(key, data=libero_like_obs[key])
        


In [17]:
ds = list(new_data_file['data'].keys())
len(ds)

70

In [20]:
new_data_file['data/demo_0']['obs'].keys()

<KeysViewHDF5 ['agentview_rgb', 'ee_ori', 'ee_pos', 'ee_states', 'eye_in_hand_rgb', 'gripper_states', 'joint_states']>

In [21]:
new_data_file.close()